# Logistic Regression – Student Pass/Fail Prediction

Loads `data.csv` if available (expected columns: studytime, failures, absences, G1, G2, G3). Otherwise uses generated sample student data so the notebook runs independently.

## Anaconda / Jupyter Notebook
Run the cells from top to bottom. If a required package is missing, run `%pip install package-name` in a notebook cell, then restart the kernel if needed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

try:
    df = pd.read_csv("data.csv")
    required = ["studytime", "failures", "absences", "G1", "G2", "G3"]
    if not set(required).issubset(df.columns):
        raise ValueError("data.csv must contain: " + ", ".join(required))
    print("Loaded data.csv")
except FileNotFoundError:
    rng = np.random.default_rng(42)
    n = 120
    g1 = rng.integers(0, 21, n)
    g2 = np.clip(g1 + rng.integers(-4, 5, n), 0, 20)
    df = pd.DataFrame({
        "studytime": rng.integers(1, 5, n),
        "failures": rng.integers(0, 4, n),
        "absences": rng.integers(0, 20, n),
        "G1": g1, "G2": g2,
        "G3": np.clip((g1 + g2) / 2 + rng.normal(0, 3, n), 0, 20).round()
    })
    print("data.csv not found. Using generated sample student data.")

df["passed"] = (df["G3"] >= 10).astype(int)
print("Dataset preview:")
display(df.head())


In [ ]:
feature_cols = ["studytime", "failures", "absences", "G1", "G2"]
X = df[feature_cols]
y = df["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"], zero_division=0))


In [ ]:
# Logistic probability curve using G2 as the only feature
single_model = LogisticRegression(max_iter=1000)
single_model.fit(df[["G2"]], df["passed"])

g2_range = np.linspace(0, 20, 300).reshape(-1, 1)
probabilities = single_model.predict_proba(g2_range)[:, 1]

plt.figure(figsize=(9, 5))
plt.plot(g2_range, probabilities, label="Probability of passing")
plt.scatter(df["G2"], df["passed"], alpha=0.5, label="Actual class")
plt.axhline(0.5, linestyle="--", label="Decision threshold = 0.5")
plt.xlabel("G2 Grade")
plt.ylabel("Probability of Passing")
plt.title("Logistic Regression: Pass/Fail from G2")
plt.grid(True)
plt.legend()
plt.show()
